# LLMのAPIの基礎


## 動作確認

In [ ]:
!python --version

In [ ]:
print("hello")

## Chat Completions API を試してみよう！


### OpenAI API キー（環境変数）の設定


In [ ]:
# このコードを実行する前に...
# `.env.template` ファイルをコピーして `.env` ファイルを作成してください。
# `.env` ファイルには OpenAI API キーを記載してください。

from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

In [ ]:
import os

print(os.environ["OPENAI_API_KEY"][:3])
# "sk-" と表示されれば、OpenAIのAPIキーを環境変数に設定できています

### Chat Completions API の呼び出し


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
    reasoning_effort="minimal",  # この指定については後ほど解説します
)
print(response.to_json(indent=2))

### 会話履歴を踏まえた応答を得る


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
        {
            "role": "assistant",
            "content": "こんにちは、ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししましょうか？",
        },
        {"role": "user", "content": "私の名前が分かりますか？"},
    ],
    reasoning_effort="minimal",
)
print(response.to_json(indent=2))

### ストリーミングで応答を得る


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "自己紹介してください。"},
    ],
    stream=True,
    reasoning_effort="minimal",
)

for chunk in response:
    if len(chunk.choices) == 0:
        continue
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)

## Vision（画像入力）


In [ ]:
from openai import OpenAI

client = OpenAI()

image_url = "https://raw.githubusercontent.com/GenerativeAgents/agent-book/refs/heads/main/assets/cover.jpg"

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "画像を説明してください。"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
    reasoning_effort="minimal",
)

print(response.choices[0].message.content)

## LangChainの「Model」


### LangChainでのLLMの呼び出し


In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！私はジョンと言います"),
    AIMessage("こんにちは、ジョンさん！どのようにお手伝いできますか？"),
    HumanMessage("私の名前がわかりますか？"),
]

ai_message = model.invoke(messages)
print(ai_message.content)

### ストリーミング


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("こんにちは！"),
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

# プロンプトの構成要素の基礎


## Few-shot プロンプティング


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "入力がAIに関係するかどうかを判断して「ＴＲＵＥ」または「ＦＡＬＳＥ」で端的に回答してください。",
        },
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "入力がAIに関係するかどうかを判断して「ＴＲＵＥ」または「ＦＡＬＳＥ」で端的に回答してください。",
        },
        {"role": "user", "content": "AIの進化はすごい"},
        {"role": "assistant", "content": "ＴＲＵＥ"},
        {"role": "user", "content": "今日は良い天気だ"},
        {"role": "assistant", "content": "ＦＡＬＳＥ"},
        {"role": "user", "content": "LLMの進化は早い"},
        {"role": "assistant", "content": "ＴＲＵＥ"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

## Zero-shot Chain of Thought プロンプティング


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "ステップバイステップで考えてください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="minimal",
)
print(response.choices[0].message.content)

## Reasoning モデル


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "developer", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
    reasoning_effort="high",  # Reasoningを最大に
)
print(response.choices[0].message.content)